In [0]:
 %sql
 CREATE VOLUME IF NOT EXISTS product_data_volume

In [0]:
 import requests

 # Download the CSV file
 url = "https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/products.csv"
 response = requests.get(url)
 response.raise_for_status()

 # Get the current catalog
 catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

 # Write directly to Unity Catalog volume
 volume_path = f"/Volumes/{catalog_name}/default/product_data_volume/products.csv"
 with open(volume_path, "wb") as f:
     f.write(response.content)

In [0]:
df = spark.read.load(volume_path, format='csv', header=True)
display(df.limit(10))

ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3399.9900
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.9900
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.9900
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.9900
775,"Mountain-100 Black, 38",Mountain Bikes,3374.9900
776,"Mountain-100 Black, 42",Mountain Bikes,3374.9900
777,"Mountain-100 Black, 44",Mountain Bikes,3374.9900
778,"Mountain-100 Black, 48",Mountain Bikes,3374.9900
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.9900
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.9900


In [0]:
 # Create the table if it does not exist. Otherwise, replace the existing table.
 df.writeTo("Products").createOrReplace()

In [0]:
 from delta.tables import DeltaTable

 # Reference the Delta table in Unity Catalog
 deltaTable = DeltaTable.forName(spark, "Products")

 # Perform the update
 deltaTable.update(
     condition = "ProductID == 771",
     set = { "ListPrice": "ListPrice * 0.9" })

 # View the updated data as a dataframe
deltaTable.toDF().filter("ProductID = 771").show()


+---------+--------------------+--------------+----------+
|ProductID|         ProductName|      Category| ListPrice|
+---------+--------------------+--------------+----------+
|      771|Mountain-100 Silv...|Mountain Bikes|2478.59271|
+---------+--------------------+--------------+----------+



In [0]:
 new_df = spark.read.option("versionAsOf", 0).table("Products")
 new_df.show(10)

+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      772|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      773|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      774|Mountain-100 Silv...|Mountain Bikes|3399.9900|
|      775|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      776|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      777|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      778|Mountain-100 Blac...|Mountain Bikes|3374.9900|
|      779|Mountain-200 Silv...|Mountain Bikes|2319.9900|
|      780|Mountain-200 Silv...|Mountain Bikes|2319.9900|
+---------+--------------------+--------------+---------+
only showing top 10 rows


In [0]:
deltaTable.history(10).show(10, False, True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 version             | 11                                                                                                                                                                                                                                                                                                                             
 timestamp           | 2026-02-01 21:17:52                                                                                                                                                                                                                                                                                

In [0]:
 spark.sql("OPTIMIZE Products")
 spark.sql("VACUUM Products RETAIN 168 HOURS") # 7 days

DataFrame[path: string]